# 09 — Modelo de duração da corrida (feature criada)

**Problema:** a duração só é conhecida depois que a corrida termina, mas é uma informação importante para o preço e para validar a hipótese de negócio. Este notebook cria um modelo que **prevê a duração antes da corrida**, usando apenas origem, destino e horário da solicitação. A saída vira a feature `duracao_estimada` do modelo de preço (notebook 11).

**Fluxo:** Bronze → features pré-corrida (`00_features_corrida`) → tabela Gold `desloc.gold.corridas_features` → comparação de modelos com validação cruzada → previsões **out-of-fold** em `desloc.gold.duracao_oof` → registro do modelo `desloc.models.duracao_corrida` no Unity Catalog.

In [ ]:
%run ./00_features_corrida

## 1. Base com o horário da solicitação

`datahora_inicio = datahora_entregue − duracao` aproxima o momento em que o passageiro pede a corrida. As features são calculadas pela mesma função que será usada na inferência.

In [ ]:
import mlflow
mlflow.autolog(disable=True)
mlflow.set_registry_uri("databricks-uc")

bronze = spark.sql("""
    SELECT to_timestamp(datahora_entregue, 'dd/MM/yyyy HH:mm') AS datahora_entregue,
           CAST(duracao AS DOUBLE) AS duracao,
           CAST(vlr_pago AS DOUBLE) AS vlr_pago,
           CAST(latitude_origem AS DOUBLE) AS latitude_origem,
           CAST(longitude_origem AS DOUBLE) AS longitude_origem,
           CAST(latitude_destino AS DOUBLE) AS latitude_destino,
           CAST(longitude_destino AS DOUBLE) AS longitude_destino
    FROM desloc.bronze.transacoes
""").toPandas()

bronze["datahora_inicio"] = bronze["datahora_entregue"] - pd.to_timedelta(bronze["duracao"], unit="m")
bronze["id_corrida"] = range(1, len(bronze) + 1)
base = pd.concat([bronze, preparar_features(bronze, "datahora_inicio")], axis=1)
print(f"Corridas: {len(base)}")
display(base.head())

## 2. Outlier de duração

O `04_eda` já destacava corridas com `duracao > 50`. Há uma corrida de **106 min** para cerca de 4,9 km, um valor implausível para o trajeto (provável erro de registro ou corrida parada). Ela fica fora do treino da duração e do preço e é marcada na tabela Gold com `outlier_duracao = true`, para ficar rastreável.

In [ ]:
LIMITE_DURACAO_MIN = 50
base["outlier_duracao"] = base["duracao"] > LIMITE_DURACAO_MIN
display(base[base.outlier_duracao][["id_corrida", "duracao", "distancia_lalo", "vlr_pago"]])

spark.createDataFrame(base).write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("desloc.gold.corridas_features")
print("Tabela desloc.gold.corridas_features gravada")

treino = base[~base.outlier_duracao].reset_index(drop=True)
X, y = treino[FEATURES_DURACAO], treino["duracao"]
print(f"Corridas no treino: {len(treino)}")

## 3. Comparação de modelos (validação cruzada 5-fold)

Com cerca de 100 corridas, uma única divisão treino/teste seria instável. Por isso a escolha usa a média de 5 dobras. A **média (Dummy)** é a referência mínima: um modelo útil precisa ter erro menor que ela.

In [ ]:
from sklearn.model_selection import KFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

SEMENTE = 42
cv = KFold(n_splits=5, shuffle=True, random_state=SEMENTE)
candidatos = {
    "media_dummy": DummyRegressor(strategy="mean"),
    "regressao_linear": Pipeline([("escala", StandardScaler()), ("modelo", LinearRegression())]),
    "knn_5": Pipeline([("escala", MinMaxScaler()), ("modelo", KNeighborsRegressor(n_neighbors=5))]),
    "random_forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=SEMENTE),
    "gradient_boosting": GradientBoostingRegressor(n_estimators=200, max_depth=2, learning_rate=0.05, random_state=SEMENTE),
}

linhas = []
for nome, modelo in candidatos.items():
    r = cross_validate(modelo, X, y, cv=cv, scoring=("neg_mean_absolute_error", "neg_root_mean_squared_error"))
    linhas.append({"modelo": nome,
                   "MAE_min": -r["test_neg_mean_absolute_error"].mean(),
                   "MAE_desvio": r["test_neg_mean_absolute_error"].std(),
                   "RMSE_min": -r["test_neg_root_mean_squared_error"].mean()})
comparacao = pd.DataFrame(linhas).sort_values("MAE_min").reset_index(drop=True)
display(comparacao.round(3))

MELHOR = comparacao.loc[comparacao.modelo != "media_dummy", "modelo"].iloc[0]
mae_dummy = comparacao.set_index("modelo").loc["media_dummy", "MAE_min"]
mae_melhor = comparacao.set_index("modelo").loc[MELHOR, "MAE_min"]
print(f"Modelo escolhido: {MELHOR} | MAE {mae_melhor:.2f} min vs média {mae_dummy:.2f} min "
      f"({100 * (1 - mae_melhor / mae_dummy):.1f}% melhor)")

## 4. Previsões out-of-fold (OOF)

Se o modelo de preço fosse treinado com a duração prevista por um modelo que **já viu aquela corrida**, a feature seria otimista demais e o ganho no preço pareceria maior do que é. Com `cross_val_predict`, cada corrida recebe a previsão de um modelo treinado **sem ela**. Essas previsões OOF são gravadas em `desloc.gold.duracao_oof` e usadas no notebook 11.

In [ ]:
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error

def root_mean_squared_error(y, y_hat):
    return float(np.sqrt(mean_squared_error(y, y_hat)))

modelo_duracao = clone(candidatos[MELHOR])
treino["duracao_estimada"] = cross_val_predict(modelo_duracao, X, y, cv=cv)
print(f"OOF — MAE {mean_absolute_error(y, treino.duracao_estimada):.2f} min | "
      f"RMSE {root_mean_squared_error(y, treino.duracao_estimada):.2f} min")

oof = treino[["id_corrida", "duracao", "duracao_estimada"]].assign(modelo=MELHOR)
spark.createDataFrame(oof).write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("desloc.gold.duracao_oof")
print("Tabela desloc.gold.duracao_oof gravada")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].scatter(treino.duracao, treino.duracao_estimada, alpha=.7)
lim = [treino.duracao.min() - 1, treino.duracao.max() + 1]
ax[0].plot(lim, lim, "k--", lw=.8)
ax[0].set(title="Duração real x estimada (OOF)", xlabel="Real (min)", ylabel="Estimada (min)")
ax[1].scatter(treino.distancia_lalo, treino.duracao, alpha=.7, label="real")
ax[1].scatter(treino.distancia_lalo, treino.duracao_estimada, alpha=.7, label="estimada")
ax[1].set(title="Duração x distância", xlabel="Distância haversine (km)", ylabel="min"); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. Registro do modelo de duração no Unity Catalog

O modelo final é treinado com todas as corridas válidas e registrado como `desloc.models.duracao_corrida`, com métricas de CV, assinatura e exemplo de entrada. O notebook 11 o incorpora ao modelo encadeado que vai para o Serving.

In [ ]:
import inspect
from mlflow.models import infer_signature

usuario = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{usuario}/desloc-duracao-corrida")
NOME_MODELO_DURACAO = "desloc.models.duracao_corrida"

modelo_duracao.fit(X, y)
exemplo = X.head(5)
with mlflow.start_run(run_name=f"duracao-{MELHOR}") as run:
    mlflow.log_params({"modelo": MELHOR, "features": ",".join(FEATURES_DURACAO),
                       "limite_outlier_min": LIMITE_DURACAO_MIN, "corridas_treino": len(X)})
    mlflow.log_metrics({"cv_mae_min": mae_melhor, "cv_mae_dummy_min": mae_dummy,
                        "oof_mae_min": mean_absolute_error(y, treino.duracao_estimada),
                        "oof_rmse_min": root_mean_squared_error(y, treino.duracao_estimada)})
    mlflow.log_table(comparacao, "comparacao_modelos.json")
    argumentos = {"sk_model": modelo_duracao,
                  "signature": infer_signature(exemplo, modelo_duracao.predict(exemplo)),
                  "input_example": exemplo.head(1),
                  "registered_model_name": NOME_MODELO_DURACAO}
    parametro = "name" if "name" in inspect.signature(mlflow.sklearn.log_model).parameters else "artifact_path"
    argumentos[parametro] = "model"
    mlflow.sklearn.log_model(**argumentos)
    RUN_ID_DURACAO = run.info.run_id
print(f"Run: {RUN_ID_DURACAO} | Modelo: {NOME_MODELO_DURACAO}")

## 6. Interpretação

- O modelo de duração reduz o erro em relação à média, mas o ganho é **moderado**: a duração depende de trânsito, paradas e rota, e nada disso está nos dados. A distância em linha reta e o horário explicam só parte da variação.
- A previsão é usada como **feature** no preço (notebook 11) e mostrada ao passageiro no app como tempo estimado.
- Para melhorar: distância e tempo de rota por ruas (por exemplo, OSRM), dados de trânsito e mais corridas.